## Bijdragen en GenAI

**Jorre Van Dyck:** dit Mushroom-model op AWS SageMaker getraind en de run-/tuningexports aangeleverd. **Codex:** notebookcode voorbereid en de aangeleverde pipeline en resultaten gecontroleerd.

Jorres bevestigde projectbijdragen: pipeline, gezamenlijke backend, UI en AWS-training. Zie [bijdragen](../BIJDRAGEN.md) en [GenAI Disclosure](../GENAI_DISCLOSURE.md), inclusief vijf echte prompts en uitgevoerde controles. De overige teamleden bevestigen hun eigen aandeel en zelfstandige beheersing.


Final-leesversie; oorspronkelijke bestanden en hun uitvoer blijven behouden. Jorres bijdrage en de GenAI-policy zijn nu verwerkt in de centrale verantwoording. Bron: `SolutionJorre/MushroomDataset/correct_dataset/04_aws_correct_dataset.ipynb`.

# AWS Mushroom — correct cursusbestand (5.000 rijen)

Jorre Van Dyck: AWS-training; Codex: voorbereid vervangingsnotebook op verzoek van
Jorre, 14 oktober 2026. **Deze aangeleverde run is uitgevoerd op SageMaker en afzonderlijk geaudit.** Upload het CSV-bestand
uit `MushroomFinal`. Dit notebook vervangt voor de inlevering de eerdere training op
61.069 UCI-rijen. `jumbled_noise_0` wordt uitgesloten; `jumbled_noise_1` blijft behouden.
Het notebook maakt geen endpoint aan.

## 1. Upload en controleer de juiste bron
Upload mushroom_project_dataset.csv naar dezelfde Jupyter-map als dit notebook. De bronhash voorkomt dat de verkeerde dataset wordt gebruikt.

In [4]:
from pathlib import Path
from urllib.request import urlretrieve

url = (
    "https://raw.githubusercontent.com/"
    "JorreVanDyck10/DeepLearningTeam10/main/"
    "MushroomFinal/mushroom_project_dataset.csv"
)

urlretrieve(url, "mushroom_project_dataset.csv")
print("Opgeslagen:", Path("mushroom_project_dataset.csv").resolve())

Opgeslagen: /home/ec2-user/SageMaker/mushroom_project_dataset.csv


In [1]:
%pip install scikit-learn==1.4.2 numpy==1.26.4 pandas==2.1.4 scipy==1.11.4 joblib==1.3.2

Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path
import hashlib, json, platform
import numpy as np
import pandas as pd
import sklearn, joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
source=Path('mushroom_project_dataset.csv')
if not source.exists():
    source=Path('../../MushroomFinal/mushroom_project_dataset.csv')
source_hash=hashlib.sha256(source.read_bytes().replace(b'\r\n',b'\n')).hexdigest()
assert source_hash=='1f1a25f2f330458ed95ce9f6ffe3241582312a42cb21797a2b7bd5e5ad281114'
data=pd.read_csv(source)
assert data.shape==(5000,13)
data.columns=data.columns.str.strip().str.lower().str.replace('-','_',regex=False)
X=data.drop(columns=['class','jumbled_noise_0'])
y=data['class'].map({'e':0,'p':1})
assert X.shape==(5000,11) and 'jumbled_noise_1' in X
print(X.shape, y.value_counts())

(5000, 11) class
0    3107
1    1893
Name: count, dtype: int64


## 2. Dezelfde splits en volledige pipeline
Gebruik dezelfde 4.100/900 rijen en vijf trainingsfolds als de teamvergelijking. Imputatie en encoding worden per trainingsfold geleerd.

In [6]:
train,test=train_test_split(np.arange(len(X)),test_size=.18,random_state=42,stratify=y)
assert len(train)==4100 and len(test)==900
numeric=X.select_dtypes(include='number').columns.tolist()
categorical=X.select_dtypes(exclude='number').columns.tolist()
preprocessor=ColumnTransformer([
    ('num',SimpleImputer(strategy='median'),numeric),
    ('cat',Pipeline([('impute',SimpleImputer(strategy='constant',fill_value='missing')),
        ('encode',OneHotEncoder(handle_unknown='ignore'))]),categorical)])
pipeline=Pipeline([('preprocessor',preprocessor),('classifier',RandomForestClassifier(random_state=42,n_jobs=2))])
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
# Four declared candidates; select only on training CV macro F1.
parameters=[
    {'classifier__n_estimators':[100],'classifier__max_depth':[None],'classifier__min_samples_leaf':[1]},
    {'classifier__n_estimators':[100],'classifier__max_depth':[24],'classifier__min_samples_leaf':[2]},
    {'classifier__n_estimators':[200],'classifier__max_depth':[16],'classifier__min_samples_leaf':[2]},
    {'classifier__n_estimators':[200],'classifier__max_depth':[None],'classifier__min_samples_leaf':[1]}]
search=GridSearchCV(pipeline,parameters,scoring='f1_macro',cv=cv,n_jobs=2,refit=True)
search.fit(X.iloc[train],y.iloc[train])
print(search.best_params_,search.best_score_)

{'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 100} 0.7531237776573464


## 3. Evalueer en exporteer
De testset is al bekend uit het teamonderzoek; benoem dit als ontwikkeltest. Download na uitvoering deze notebook en de drie correct_*-exports.

In [7]:
model=search.best_estimator_
pred=model.predict(X.iloc[test]);prob=model.predict_proba(X.iloc[test])[:,list(model.classes_).index(1)]
results={'source':'mushroom_project_dataset.csv','source_sha256_lf':source_hash,
    'rows':5000,'train_rows':4100,'test_rows':900,'features':X.columns.tolist(),
    'removed':['jumbled_noise_0'],'class_encoding':{'e':0,'p':1},
    'best_parameters':search.best_params_,'cv_macro_f1':float(search.best_score_),
    'accuracy':float(accuracy_score(y.iloc[test],pred)),
    'balanced_accuracy':float(balanced_accuracy_score(y.iloc[test],pred)),
    'macro_f1':float(f1_score(y.iloc[test],pred,average='macro')),
    'precision_p':float(precision_score(y.iloc[test],pred,zero_division=0)),
    'recall_p':float(recall_score(y.iloc[test],pred,zero_division=0)),
    'f1_p':float(f1_score(y.iloc[test],pred,zero_division=0)),
    'roc_auc':float(roc_auc_score(y.iloc[test],prob)),
    'confusion_matrix':confusion_matrix(y.iloc[test],pred,labels=[0,1]).tolist(),
    'versions':{'python':platform.python_version(),'sklearn':sklearn.__version__,'pandas':pd.__version__,'numpy':np.__version__,'joblib':joblib.__version__},
    'execution_environment':'AWS SageMaker - record only after actual execution'}
Path('correct_aws_results.json').write_text(json.dumps(results,indent=2)+'\n')
pd.DataFrame(search.cv_results_).to_csv('correct_aws_tuning_results.csv',index=False)
joblib.dump(model,'correct_mushroom_pipeline.joblib',compress=3)
print(json.dumps(results,indent=2))

{
  "source": "mushroom_project_dataset.csv",
  "source_sha256_lf": "1f1a25f2f330458ed95ce9f6ffe3241582312a42cb21797a2b7bd5e5ad281114",
  "rows": 5000,
  "train_rows": 4100,
  "test_rows": 900,
  "features": [
    "cap_diameter",
    "stem_height",
    "stem_width",
    "spore_print_color",
    "gill_color",
    "habitat",
    "season",
    "ring_type",
    "cap_shape",
    "stem_surface",
    "jumbled_noise_1"
  ],
  "removed": [
    "jumbled_noise_0"
  ],
  "class_encoding": {
    "e": 0,
    "p": 1
  },
  "best_parameters": {
    "classifier__max_depth": null,
    "classifier__min_samples_leaf": 1,
    "classifier__n_estimators": 100
  },
  "cv_macro_f1": 0.7531237776573464,
  "accuracy": 0.8011111111111111,
  "balanced_accuracy": 0.7598376866944009,
  "macro_f1": 0.7725344128182718,
  "precision_p": 0.8375,
  "recall_p": 0.5894428152492669,
  "f1_p": 0.6919104991394148,
  "roc_auc": 0.8457262917127885,
  "confusion_matrix": [
    [
      520,
      39
    ],
    [
      140,
      